In [162]:
import os
print(os.getcwd())
import sys
from pathlib import Path
import src.sequence_utils as su

print(su.__file__)

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import src.sequence_utils as su

print(hasattr(su, "sequence_gc_content"))

import importlib

importlib.reload(su)

print(hasattr(su, "sequence_gc_content"))

/public2/home/chenhaiyao2/bioai/rnsaq-cancer-classification/notebooks
/public2/home/chenhaiyao2/bioai/rnsaq-cancer-classification/src/sequence_utils.py
True
True


In [163]:
## ecercise 1: modularise the Day 1 functionality
# Goal: learn how to write function
# check if a given DNA sequence is valid
def is_valid_dna(seq):
    valid_nucleotides = set("ATCG")
    for nucleotide in seq:
        if nucleotide not in valid_nucleotides:
            return False
    return True

# Result:
assert is_valid_dna("AGCTAGC") == True
assert is_valid_dna("ATGCXYZ") == False

# calculate the GC content of a given sequence
def gc_content(seq):
    gc_count = seq.count("G") + seq.count("C")
    gc_percentage = gc_count / len(seq) * 100
    return round(gc_percentage, 2)

# Result:
assert gc_content("AGCTAGC") == 57.14

# calculate the reverse complement of a given sequence
def reverse_complement(seq):
    complement = {'A': 'T', 'T':'A', 'C':'G', 'G':'C'}
    rev_comp = ''.join(complement[nucleotide] for nucleotide in reversed(seq))
    return rev_comp

# Result:
assert reverse_complement("ATGC") == "GCAT"

# find motifs in a given sequence
def find_motif(seq, motif):
    positions = []
    motif_length = len(motif)
    for i in range(len(seq) - motif_length +1):
        if seq[i:i+motif_length] == motif:
            positions.append(i)
    return positions

# Result:
assert find_motif("ATGCGATGACCATGGCTA", "ATG") == [0, 5, 11]

In [164]:
## excercise 2: robustness and error handling
# Goal: learn how to make functions robust
def gc_content_robust(seq):
    if len(seq) == 0:
        return 0.0
    else:
        seq = seq.upper()
    return gc_content(seq)

# Result:
assert gc_content_robust("") == 0.0
assert gc_content_robust("AGCTagc") == 57.14

In [165]:
## excercise 3: open a file and read a sequence from it
## excercise 4: write a summary of the sequence to a file
# Goal: learn how to read from a file
with open("../data/example_sequence.txt", "r") as f_in:
    sequence = f_in.read().strip()

    with open("../results/day2_sequence_summary.txt", "w"
    ) as f_out:
     f_out.write(f"sequence: {sequence}\n")
     f_out.write(f"sequence length: {len(sequence)}\n")
     f_out.write(f"GC content: {gc_content_robust(sequence)}\n")
     f_out.write(f"reverse complement: {reverse_complement(sequence)}\n")

In [166]:
## excercise 5: learn fasta file
with open("../data/example.fasta") as f:
    for line in f:
        if line.startswith(">"):
            header = line.strip()
        else: 
            sequence = line.strip()
            print(f"header:{header}, sequence: {sequence}")

header:>seq1, sequence: ATGCGATCGATCGTTAGCGGCTAACG
header:>seq2, sequence: ATTTGGCCATGCGCGCGCAT
header:>seq3, sequence: GGGGCCCCAAAATTTT
header:>seq4, sequence: ATGCATGCATGCATGCATGCATGC


In [167]:
## excercise 6&7: write a fasta parser function
def read_fasta(fasta_file):
    sequences = {}
    with open(fasta_file) as f:
        header = None
        seq_lines = []
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if header:
                    sequences[header] = "".join(seq_lines)
                header = line[1:]
                seq_lines = []
            else:
                seq_lines.append(line)
        if header:
            sequences[header] = "".join(seq_lines)
    return sequences

sequecnses = read_fasta("../data/example.fasta")
print(sequecnses)
print(sequecnses["seq1"])

{'seq1': 'ATGCGATCGATCGTTAGCGGCTAACG', 'seq2': 'ATTTGGCCATGCGCGCGCAT', 'seq3': 'GGGGCCCCAAAATTTT', 'seq4': 'ATGCATGCATGCATGCATGCATGC'}
ATGCGATCGATCGTTAGCGGCTAACG


In [168]:
## excercise 8: calculate the length of each sequence in a fasta file
def sequence_lengths(fasta_file):
    sequences = read_fasta(fasta_file)
    lengths = {header: len(seq) for header, seq in sequences.items()}
    return lengths

print(sequence_lengths("../data/example.fasta"))

{'seq1': 26, 'seq2': 20, 'seq3': 16, 'seq4': 24}


In [169]:
## excercise 9: calculate the GC content of each sequence in a fasta file
# def sequence_gc_content(fasta_file):
#     sequences = read_fasta(fasta_file)
#     gc_contents = {header: gc_content_robust(seq) for header, seq in sequences.items()}
#     return gc_contents

# print(sequence_gc_content("../data/example.fasta"))

In [170]:
## excricise 10: find the highest GC content sequence in a fasta file
def highest_gc_content(fasta_file):
    gc_contents = sequence_gc_content(fasta_file)
    for header, gc in gc_contents.items():
        if gc == max(gc_contents.values()):
            print(f"Highest GC sequence: {header},{gc}%")

def highest_gc_content_2(fasta_file):
    gc_contents = sequence_gc_content(fasta_file)
    max_gc = 0
    for header, gc in gc_contents.items():
        if gc > max_gc:
            max_gc = gc
            max_header = header
    print(f"Highest GC sequence: {max_header},{max_gc}%")

highest_gc_content("../data/example.fasta")
highest_gc_content_2("../data/example.fasta")

Highest GC sequence: seq2,60.0%
Highest GC sequence: seq2,60.0%


In [171]:
## excercise 11: filter sequences based on length
def filter_sequences(fasta_file):
    sequences = read_fasta(fasta_file)
    filter_sequences = {}
    for header, seq in sequences.items():
        if len(seq) > 20:
            filter_sequences[header] = seq
    return filter_sequences

print(filter_sequences("../data/example.fasta"))

{'seq1': 'ATGCGATCGATCGTTAGCGGCTAACG', 'seq4': 'ATGCATGCATGCATGCATGCATGC'}


In [173]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.sequence_utils import summary

summary(
    "../data/example.fasta",
    "../results/summary.txt"
)

All sequences are valid
Number of sequences: 4
sequences longer than 20: {'seq1': 'ATGCGATCGATCGTTAGCGGCTAACG', 'seq2': 'ATTTGGCCATGCGCGCGCAT', 'seq4': 'ATGCATGCATGCATGCATGCATGC'}
summary written to ../results/summary.txt
